## Cargo las librerias

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Cargo los set de datos

In [8]:
df_personas = pd.read_csv(
    "H224.csv",
    sep=";",
    low_memory=False
)

df_condiciones = pd.read_csv(
    "h222.csv",
    sep=";",
    low_memory=False
)

df_medicamentos = pd.read_csv(
    "h229a.csv",
    sep=";",
    low_memory=False
)

## verifico

In [4]:
df_personas.shape

(27805, 1451)

In [5]:
df_condiciones.shape

(80802, 30)

In [6]:
df_medicamentos.shape

(284375, 66)

## Selecciono Variables utiles
Estas variables existen en H224 y son relevantes para el proyecto

In [9]:
variables_personas = [
    "DUPERSID",
    "AGE20X",
    "SEX",
    "MARRY20X",
    "EDUCYR",
    "POVCAT20",
    "EMPST53",
    "INSCOV20",
    "MNHLTH53",
    "RTHLTH53",
    "RACETHX"
]

df_personas = df_personas[variables_personas]

## Exploracion 

In [8]:
df_personas.head()

,DUPERSID,AGE20X,SEX,MARRY20X,EDUCYR,POVCAT20,EMPST53,INSCOV20,MNHLTH53,RTHLTH53,RACETHX
0,2320005101,73,2,1,14,2.0,4.0,2.0,2.0,2.0,2
1,2320005102,84,1,1,12,2.0,4.0,2.0,5.0,5.0,2
2,2320006101,47,2,3,12,3.0,4.0,3.0,3.0,3.0,1
3,2320006102,22,1,5,11,1.0,1.0,3.0,1.0,1.0,1
4,2320006103,21,1,5,11,3.0,4.0,2.0,2.0,2.0,1


In [9]:
df_personas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6642 entries, 0 to 6641
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   DUPERSID  6642 non-null   int64  
 1   AGE20X    6642 non-null   int64  
 2   SEX       6642 non-null   int64  
 3   MARRY20X  6642 non-null   int64  
 4   EDUCYR    6642 non-null   int64  
 5   POVCAT20  6641 non-null   float64
 6   EMPST53   6641 non-null   float64
 7   INSCOV20  6641 non-null   float64
 8   MNHLTH53  6641 non-null   float64
 9   RTHLTH53  6641 non-null   float64
 10  RACETHX   6642 non-null   int64  
dtypes: float64(5), int64(6)
memory usage: 570.9 KB


## condiciones medicas 
Cantidad de condiciones por persona.

In [10]:
df_condiciones_persona = (
    df_condiciones
    .groupby("DUPERSID")
    .agg(
        cantidad_condiciones=("DUPERSID","count")
    )
    .reset_index()
)

df_condiciones_persona.head()

,DUPERSID,cantidad_condiciones
0,2320005101,1
1,2320005102,2
2,2320006102,1
3,2320006103,1
4,2320012102,5


## LIMPIEZA DE MEDICAMENTOS

In [13]:
df_medicamentos["RXDRGNAM"] = (
    df_medicamentos["RXDRGNAM"]
    .astype(str)
    .str.upper()
    .str.strip()
)

## CLASIFICACIÓN DE PSICOFÁRMACOS

In [14]:
benzodiacepinas = [
    "ALPRAZOLAM",
    "CLONAZEPAM",
    "DIAZEPAM",
    "LORAZEPAM",
    "TEMAZEPAM",
    "MIDAZOLAM"
]

antidepresivos = [
    "FLUOXETINE",
    "SERTRALINE",
    "PAROXETINE",
    "ESCITALOPRAM",
    "CITALOPRAM",
    "VENLAFAXINE",
    "DULOXETINE",
    "BUPROPION",
    "AMITRIPTYLINE"
]

hipnoticos = [
    "ZOLPIDEM",
    "ZALEPLON",
    "ESZOPICLONE"
]

ansioliticos = [
    "BUSPIRONE",
    "HYDROXYZINE"
]

## V. Derivadas

In [15]:
df_medicamentos["benzo"] = (
    df_medicamentos["RXDRGNAM"]
    .isin(benzodiacepinas)
    .astype(int)
)

df_medicamentos["antidepresivo"] = (
    df_medicamentos["RXDRGNAM"]
    .isin(antidepresivos)
    .astype(int)
)

df_medicamentos["hipnotico"] = (
    df_medicamentos["RXDRGNAM"]
    .isin(hipnoticos)
    .astype(int)
)

df_medicamentos["ansiolitico"] = (
    df_medicamentos["RXDRGNAM"]
    .isin(ansioliticos)
    .astype(int)
)

## AGR. DE PSICOFÁRMACOS

In [16]:
df_psicofarmacos = (
    df_medicamentos
    .groupby("DUPERSID")
    .agg(
        recetas_totales=("RXDRGNAM","count"),
        benzodiacepinas=("benzo","sum"),
        antidepresivos=("antidepresivo","sum"),
        hipnoticos=("hipnotico","sum"),
        ansioliticos=("ansiolitico","sum")
    )
    .reset_index()
)

df_psicofarmacos["total_psicofarmacos"] = (
    df_psicofarmacos["benzodiacepinas"]
    + df_psicofarmacos["antidepresivos"]
    + df_psicofarmacos["hipnoticos"]
    + df_psicofarmacos["ansioliticos"]
)

## union 

In [17]:
df_final = (
    df_personas
    .merge(
        df_condiciones_persona,
        on="DUPERSID",
        how="left"
    )
    .merge(
        df_psicofarmacos,
        on="DUPERSID",
        how="left"
    )
)

## limpio nulos

In [18]:
df_final.fillna(0, inplace=True)

## verifico

In [19]:
df_final.shape

(27805, 18)

In [20]:
df_final.head()

,DUPERSID,AGE20X,SEX,MARRY20X,EDUCYR,POVCAT20,EMPST53,INSCOV20,MNHLTH53,RTHLTH53,RACETHX,cantidad_condiciones,recetas_totales,benzodiacepinas,antidepresivos,hipnoticos,ansioliticos,total_psicofarmacos
0,2320005101,73,2,1,14,2,4,2,2,2,2,1.0,6.0,0.0,0.0,0.0,0.0,0.0
1,2320005102,84,1,1,12,2,4,2,5,5,2,2.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2320006101,47,2,3,12,3,4,3,3,3,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2320006102,22,1,5,11,1,1,3,1,1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2320006103,21,1,5,11,3,4,2,2,2,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27805 entries, 0 to 27804
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   DUPERSID              27805 non-null  int64  
 1   AGE20X                27805 non-null  int64  
 2   SEX                   27805 non-null  int64  
 3   MARRY20X              27805 non-null  int64  
 4   EDUCYR                27805 non-null  int64  
 5   POVCAT20              27805 non-null  int64  
 6   EMPST53               27805 non-null  int64  
 7   INSCOV20              27805 non-null  int64  
 8   MNHLTH53              27805 non-null  int64  
 9   RTHLTH53              27805 non-null  int64  
 10  RACETHX               27805 non-null  int64  
 11  cantidad_condiciones  27805 non-null  float64
 12  recetas_totales       27805 non-null  float64
 13  benzodiacepinas       27805 non-null  float64
 14  antidepresivos        27805 non-null  float64
 15  hipnoticos         

## LIsto , exporto

In [22]:
df_final.to_csv(
    "dataset_psicofarmacos_final.csv",
    index=False
)